In [2]:
import urllib3
import json

In [3]:
#See documentation: https://lbgi.fr/api/orthoinspector#/

In [ ]:
"""def get_orthologs(qtaxid, ttaxid):
    resp = urllib3.request("GET", f"https://lbgi.fr/api/orthoinspector/Eukaryota2023/species/{qtaxid}/orthologs/{ttaxid}",
                            timeout=urllib3.util.Timeout(600))
    return json.loads(resp.data.decode())"""

In [2]:
import json
import urllib3

def get_orthologs(qtaxid, ttaxid):
    resp = urllib3.request(
        "GET", 
        f"https://lbgi.fr/api/orthoinspector/Eukaryota2023/species/{qtaxid}/orthologs/{ttaxid}",
        timeout=urllib3.util.Timeout(600)
    )
    
    raw_data = resp.data.decode().strip()
    
    # Use raw_decode to take only the first valid JSON object found
    try:
        decoder = json.JSONDecoder()
        # raw_decode returns a tuple: (the_parsed_object, index_where_it_stopped)
        data, index = decoder.raw_decode(raw_data)
        return data
    except json.JSONDecodeError as e:
        print(f"Actual error: {e}")
        # If it fails here, the data itself is likely truncated/broken
        raise e

In [1]:
import json
import urllib3

def get_protein(taxid):
    resp = urllib3.request(
        "GET", 
        f"https://lbgi.fr/api/orthoinspector/Eukaryota2023/species/{taxid}/proteins",
        timeout=urllib3.util.Timeout(600)
    )
    
    raw_data = resp.data.decode().strip()
    
    try:
        # On utilise le décodeur brut pour s'arrêter dès que le premier JSON est fini
        decoder = json.JSONDecoder()
        data, index = decoder.raw_decode(raw_data)
        return data
    except json.JSONDecodeError as e:
        print(f"Erreur de décodage pour le taxid {taxid}: {e}")
        return None

In [2]:
def extract_pairs(json_data):
    all_opairs = []
    for data in json_data['data']:
        for ip in data.get("inparalogs",[]):
            for ot in data.get("orthologs",[]):
                all_opairs.append((ip,ot))
    return all_opairs

def write_data(all_pairs, filename):
    with open(filename,'w') as handle:
        for pair in all_pairs:
            handle.write("\t".join(pair)+"\n")

In [3]:
def extract_protein_ids(json_data):
    ids = []
    for item in json_data.get('data', []):
        protein_id = item.get('access') 
        if protein_id:
            ids.append([protein_id]) 
    return ids

In [16]:
human_mouse = get_orthologs(9606,559292)
orthopairs = extract_pairs(human_mouse)
write_data(orthopairs, "/home/cassandre/stage/Cassandre/ortho/liste_orthologs/zebrafish_HUMAN.tsv")

MaxRetryError: HTTPSConnectionPool(host='lbgi.fr', port=443): Max retries exceeded with url: /api/orthoinspector/Eukaryota2023/species/9606/orthologs/559292 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='lbgi.fr', port=443): Read timed out. (read timeout=599.9753121295944)"))

In [4]:
protein_homosapiens = get_protein(7227)

if protein_homosapiens and "data" in protein_homosapiens:
    # On transforme le JSON en liste d'IDs
    protein_list = extract_protein_ids(protein_homosapiens)
    
    # On écrit dans ton dossier
    path = "/home/cassandre/stage/Cassandre/ortho/list_protein_orthoinspector/droso_protein.tsv"
    write_data(protein_list, path)
    
    print(f"Succès ! {len(protein_list)} protéines enregistrées.")
else:
    print("Erreur : La réponse de l'API est vide ou mal formatée.")

Succès ! 13821 protéines enregistrées.
